# pitchlake · run pipeline

Parameterised entry point used by the **pl_statsbomb_daily** Data Pipeline (one activity per stage)
and for ad-hoc runs. Attach the **lh_football** lakehouse as the default lakehouse.

| layer | where |
|---|---|
| bronze | `Files/bronze/statsbomb/<entity>/<partition>/<key>/<ts>_<sha>.json` |
| silver / gold / meta | `Tables/silver.*`, `Tables/gold.*`, `Tables/meta.*` (Delta) |

In [ ]:
# ---- parameters (overridden by the Data Pipeline activity) ----
steps = "bronze,silver,synthetic,integrate,dq_silver,gold,dq_gold"   # comma-separated
scope = "55:282"                 # competition_id:season_id[,cid:sid...]
max_matches = 0                  # 0 = no limit
include_360 = True
full_refresh = False
dq_mode = "enforce"              # enforce | warn
environment = "fabric"           # fabric | fabric-prod
# Pinned release of the framework. A GitHub tag archive needs no git client in the runtime.
# Alternative: upload the wheel from CI to Files/libs and use "/lakehouse/default/Files/libs/pitchlake-1.0.0-py3-none-any.whl"
package_source = "https://github.com/toxifarian/mfc-statsbomb-lakehouse/archive/refs/tags/v1.0.0.zip"

In [ ]:
# Install the framework at a pinned, tagged version (same code as CI tested).
%pip install -q "{package_source}"

In [ ]:
import json, os
from pitchlake.ops.logging_setup import configure_logging
from pitchlake.pipeline import Pipeline

configure_logging("INFO")
overrides = {
    "sources": {"statsbomb": {
        "scope": [{"competition_id": int(c), "season_id": int(s), "label": f"{c}:{s}"}
                  for c, s in (x.split(":") for x in scope.split(","))],
        "max_matches_per_run": int(max_matches) or None,
        "entities": {"lineups": True, "events": True, "frames_360": bool(include_360)},
    }},
    "quality": {"mode": dq_mode},
}
pipe = Pipeline.from_env(environment, overrides=overrides)
print(pipe.platform.describe())
ctx = pipe.run(steps=[s.strip() for s in steps.split(",") if s.strip()],
               full_refresh=bool(full_refresh), trigger="fabric")
summary = {"run_id": ctx.run_id, "steps": ctx.summary()}
print(json.dumps(summary, indent=2, default=str))

In [ ]:
# Return a compact result to the calling Data Pipeline (visible in the activity output)
notebookutils.notebook.exit(json.dumps({"run_id": ctx.run_id,
                                        "failed": [s["step"] for s in ctx.summary() if s["status"] != "succeeded"]}))